###Part A — Bigram Baseline

Time: about 3 hours.

Build a character-level bigram language model in a single file, task1/bigram.py. A bigram model
predicts the next character based only on the current character — no context. The model is literally a
lookup table of shape (vocab_size, vocab_size).

In [4]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import os

# -----------------------------------------------------------------------------
# Hyperparameters
batch_size = 32
block_size = 8
max_iters = 3000
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_interval = 300
# -----------------------------------------------------------------------------

torch.manual_seed(1337)

# 4. Read the text file and build the vocabulary
file_path = '/content/input.txt'
with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size}")

# 5. Build the tokenizer
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# 6. Encode the entire text into a tensor
data = torch.tensor(encode(text), dtype=torch.long)

# 7. Split 90/10 into train and validation tensors
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# 8. Write get_batch(split)
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data_source = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))
    x = torch.stack([data_source[i:i+block_size] for i in ix])
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(200)
        for k in range(200):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# 9. Build the BigramLanguageModel
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    # 11. Implement generate using categorical sampling
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel(vocab_size)
model = model.to(device)

# 10. Training loop
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("Starting training...")
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Generate sample text after training
print("\n--- Generating Text ---")
context = torch.zeros((1, 1), dtype=torch.long, device=device) # start with the newline character
generated_chars = decode(model.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

Vocabulary size: 65
Starting training...
step 0: train loss 4.7305, val loss 4.7241
step 300: train loss 2.8110, val loss 2.8249
step 600: train loss 2.5434, val loss 2.5682
step 900: train loss 2.4932, val loss 2.5088
step 1200: train loss 2.4863, val loss 2.5035
step 1500: train loss 2.4665, val loss 2.4921
step 1800: train loss 2.4683, val loss 2.4936
step 2100: train loss 2.4696, val loss 2.4846
step 2400: train loss 2.4638, val loss 2.4879
step 2700: train loss 2.4738, val loss 2.4911
step 2999: train loss 2.4611, val loss 2.4903

--- Generating Text ---

LIZAntaitoupis!
BENIngt,
N tiel, serhe hill: h wous sal ayolf sthereeyowoulour: horgonof m


sunicour,


ANLOurak anominfaind oul bond f DIC:
O g.

Gr IOLouspold se.
Dotamy t Y mioke om, d a he ates,
ARDus ang s tist;
's
ORINUSTod ad yety CLAns,
Takerecithak ws: got hesal tobjobis,'s dsent,
BRCOfararnt wilanou?
F ve:
ORDO
FOpe tend d hil RK:
Four wath, f.
I
BUMEToirofre ahow ive I's,
NI to'l
A:
NTINERK:
SThingou, thingilet menyo 

###🔎 CURIOSITY CORNER — Why is the starting loss log(vocab_size)?

At initialization, your model knows nothing, so each of the vocab_size outputs is roughly equally
likely. The probability of the correct token is therefore about 1/vocab_size, and the cross-entropy
loss is -log(1/vocab_size) = log(vocab_size). For tiny Shakespeare's 65 characters, that's about 4.17.
This is a useful sanity check — if your starting loss is wildly different, your initialization or your loss computation is wrong.

###🔎 CURIOSITY CORNER — AdamW versus SGD versus everything else

SGD with momentum is the classical optimizer — simple, robust, well-understood. It is still the
workhorse for image classification with CNNs. Adam adds per-parameter adaptive learning rates by
tracking running averages of gradients and their squares. AdamW corrects a subtle bug in Adam's
interaction with weight decay (decoupling decay from the gradient update). For transformers,
AdamW is the default for good reason: it handles the heterogeneous gradient magnitudes across
embedding, attention, and MLP parameters gracefully.
Worth exploring later: Lion (Google, 2023) — uses only sign of gradient and is more memory-
efficient. Sophia (Stanford, 2023) — uses a diagonal Hessian estimate for faster convergence on
LLMs. Shampoo and its variants — full second-order methods that are gaining traction at scale. For
this project, stick with AdamW.

###Part B — Adding a Single Self-Attention Head
Time: 5–6 hours.

Now you will write attention yourself. Copy bigram.py to task1/attention.py and modify.
Conceptual setup. A self-attention head takes in a sequence of token embeddings and produces a
sequence of "context-aware" embeddings. For each position t, it looks at all previous positions (because of the causal mask), decides which ones are relevant, and produces a weighted sum of their values. The mechanism by which it decides relevance is a learned similarity between queries(from position t) and keys (from all earlier positions).

In [5]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import os

# -----------------------------------------------------------------------------
# Hyperparameters
batch_size = 32
block_size = 8
max_iters = 5000
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_interval = 500
n_embd = 32      # Added: embedding dimension
head_size = 32   # Added: size of the attention head
# -----------------------------------------------------------------------------

torch.manual_seed(1337)

# 4. Read the text file and build the vocabulary
# Using the standard Colab/local path handling
file_path = 'input.txt' if os.path.exists('input.txt') else '/content/input.txt'
with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

# 5. Build the tokenizer
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data_source = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))
    x = torch.stack([data_source[i:i+block_size] for i in ix])
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(200)
        for k in range(200):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# --- NEW: The Self-Attention Head ---
class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        v = self.value(x) # (B, T, head_size)

        # attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) # (B, T, head_size) @ (B, head_size, T) ---> (B, T, T)
        wei = wei * (k.size(-1) ** -0.5) # scale by 1/sqrt(d_k)

        # causal mask: tokens can't look into the future
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1) # (B, T, T)

        # weighted sum of values
        out = wei @ v # (B, T, T) @ (B, T, head_size) ---> (B, T, head_size)
        return out

# --- UPDATED: The Main Model Architecture ---
class AttentionLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, head_size, block_size):
        super().__init__()
        # Token identities get an embedding vector
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # Token positions also get an embedding vector
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # The single self-attention head
        self.sa_head = Head(n_embd, head_size, block_size)

        # Final linear layer to decode back to vocabulary size
        self.lm_head = nn.Linear(head_size, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # x and y are (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, n_embd)

        # Add token and position embeddings together
        x = tok_emb + pos_emb # (B, T, n_embd)

        # Apply the self-attention head
        x = self.sa_head(x) # (B, T, head_size)

        # Get logits
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # CRITICAL ADDITION: crop idx to the last block_size tokens.
            # Without this, position embeddings will crash if the generated sequence exceeds block_size (8).
            idx_cond = idx[:, -block_size:]

            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] # (B, C)
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = AttentionLanguageModel(vocab_size, n_embd, head_size, block_size)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("Starting training with Single Head Attention...")
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("\n--- Generating Text ---")
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(model.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

Starting training with Single Head Attention...
step 0: train loss 4.2000, val loss 4.2047
step 500: train loss 2.6911, val loss 2.7087
step 1000: train loss 2.5196, val loss 2.5303
step 1500: train loss 2.4775, val loss 2.4829
step 2000: train loss 2.4408, val loss 2.4523
step 2500: train loss 2.4272, val loss 2.4435
step 3000: train loss 2.4130, val loss 2.4327
step 3500: train loss 2.3956, val loss 2.4212
step 4000: train loss 2.4041, val loss 2.3992
step 4500: train loss 2.3980, val loss 2.4084
step 4999: train loss 2.3951, val loss 2.4126

--- Generating Text ---

Ano' ou sene od wistwin, wildalle adery wheerel cro ove.

K:
OMInciove dan uts wat fo stu bur,
Wer
my seng
Becthikure
Pe.
INCECLIUS: MId miombellavo focen soun ers.

A:
Way berup he past arr:
A sg wou poat;
Wino-f-
I: herove, wipeofre
Yor.
Bur ugigiatns Ligo wisen't here, thee my teath, weird, ghe sss the D:
NCGas In.

WAnd, hei iett or ganger ng
Pow, yout Can werd? I, we.

Thth asthe mtolecraby st
Cow thy had ally whin 


### 1. The Bigram Model: The "Goldfish Memory" Approach

A bigram model is the simplest form of a language model. The prefix "bi" means two, and "gram" refers to the pieces of data (in your case, characters).

It predicts the next character by looking at exactly **one** preceding character. That’s it. It has zero memory of anything that happened before that single letter.

* **How it thinks:** If the current letter is "t", the model looks at its internal lookup table and says, *"In Shakespeare, what letter most frequently follows a 't'?"* It will likely predict "h", forming "th". But when it gets to "h", it completely forgets the "t". It only asks, *"What follows an 'h'?"* (probably "e").
* **In your code:** This was literally just a 65x65 table (`nn.Embedding(vocab_size, vocab_size)`). Row 'A' maps to the probabilities of all 65 characters following 'A'.
* **The limitation:** Because it only has a context window of 1, it cannot form multi-letter words or understand grammar. It can only learn character-pair frequencies (like knowing "q" is almost always followed by "u").

---

### 2. The Self-Attention Head: The "Cocktail Party" Mechanism

If the Bigram model is reading a book through a tiny straw where you can only see one letter, **Self-Attention** is taking a step back to look at the whole sentence (or up to your `block_size` of 8 characters) and figuring out which letters matter most to the current one.

Attention is simply a communication mechanism between tokens. It allows tokens to "talk" to each other and update their understanding of themselves based on their neighbors.

* **How it thinks (Query, Key, Value):** Imagine you are at a crowded cocktail party.
* **Query (What I want):** You want to find someone talking about PyTorch.
* **Key (What others offer):** You listen to the tags on everyone's conversations. Person A's key is "Cooking", Person B's key is "PyTorch".
* **Affinity (The match):** Your Query matches Person B's Key. You assign a high "attention score" to them and ignore Person A.
* **Value (The actual info):** You now take the actual information (the Value) Person B is speaking and absorb it.


* **In your code:** When trying to predict the 9th character, the 8th character emits a **Query**. All the previous 8 characters emit **Keys**. The model calculates the dot product (similarities) between the Query and all the past Keys. If the 4th character is highly relevant, the 8th character will pull a lot of information (the **Value**) from that 4th character to help predict the 9th.
* **The Mask:** In your code, you also applied a `tril` (lower triangular) mask. This is a **causal mask**, which simply enforces the rule of time: characters can only "pay attention" to characters that came *before* them. The 4th character cannot look at the 8th character, because the 8th character hasn't been generated yet.

**The TL;DR:**
A bigram model predicts the future based on a rigid, hardcoded lookup of the immediate past. A self-attention head looks at a longer stretch of the past and dynamically calculates which specific parts of that past are actually useful for predicting the future.

###🔎 CURIOSITY CORNER — Why scale by sqrt(d_k)?
If Q and K are drawn from standard normal distributions and you take their dot product over d_k
dimensions, the variance of the result grows linearly with d_k. With large d_k, the resulting
attention scores have huge magnitudes. After softmax, this produces extremely peaked
distributions where one position gets almost all the weight and everything else is essentially zero.
Gradients through that softmax become tiny.
Dividing by sqrt(d_k) keeps the variance of the dot product roughly constant regardless of
dimension, so softmax behaves well. This is one of the subtler details in the original Transformer
paper — and one of the most asked-about. You'll prove this rigorously in Task 2.